# STDL-Net Test Result Analysis
# Load best_small.pth, visualize predictions, analyze per-class performance

In [ ]:
# Cell 1: 安装依赖
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
# Cell 2: 基础设置
import os, sys, shutil
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import DataLoader

print(f'PyTorch: {torch.__version__}')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 3: 路径配置
# ========== 数据路径 ==========
DATA_ROOT = '/kaggle/input/lunar-data-v3/dataset/dataset'
CODE_ROOT = '/kaggle/input/lunar-data-v3/STDL-Net/STDL-Net'

if not os.path.isdir(DATA_ROOT):
    DATA_ROOT = '/kaggle/input/lunar-data-v2/dataset/dataset'
    CODE_ROOT = '/kaggle/input/lunar-data-v2/STDL-Net/STDL-Net'
    print('WARNING: v3 not found, falling back to v2')

# ========== 模型配置 ==========
MODEL_SIZE = 'small'

# ========== 模型权重 + 训练历史路径 ==========
WEIGHT_PATH = ''
HISTORY_PATH = ''
RESULT_DIR_INPUT = ''

# 自动搜索: 优先找 best_small.pth
print('自动搜索训练 Output...')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fp = os.path.join(root, f)
        if f == f'best_{MODEL_SIZE}.pth' and not WEIGHT_PATH:
            WEIGHT_PATH = fp
            print(f'  权重: {fp}')
        elif f == 'best_small.pth' and not WEIGHT_PATH:
            WEIGHT_PATH = fp
            MODEL_SIZE = 'small'
            print(f'  权重(fallback small): {fp}')
        if f == 'history.json' and not HISTORY_PATH:
            HISTORY_PATH = fp
            RESULT_DIR_INPUT = root
            print(f'  训练历史: {fp}')

TEST_IMG  = os.path.join(DATA_ROOT, 'test', 'image')
TEST_MASK = os.path.join(DATA_ROOT, 'test', 'mask')
PRETRAIN  = os.path.join(DATA_ROOT, 'pretrain')

SAVE_DIR = '/kaggle/working/visualization'
PRED_DIR = '/kaggle/working/all_predictions'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

print(f'\nModel size: {MODEL_SIZE}')
print(f'Test images: {len(os.listdir(TEST_IMG))} files')
print(f'Test masks:  {len(os.listdir(TEST_MASK))} files')
print(f'Weight: {WEIGHT_PATH}')
print(f'Weight exists: {os.path.exists(WEIGHT_PATH) if WEIGHT_PATH else False}')
print(f'History exists: {os.path.exists(HISTORY_PATH) if HISTORY_PATH else False}')

if not WEIGHT_PATH or not os.path.exists(WEIGHT_PATH):
    print('\nWARNING: 找不到权重文件! 所有 .pth 文件:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.pth'):
                print(f'  {os.path.join(root, f)}')

In [ ]:
# Cell 4: 复制代码 + 导入
CODE_DST = '/kaggle/working/code'
os.makedirs(CODE_DST, exist_ok=True)
for f in os.listdir(CODE_ROOT):
    if f.endswith('.py'):
        shutil.copy2(os.path.join(CODE_ROOT, f), os.path.join(CODE_DST, f))
sys.path.insert(0, CODE_DST)

# 修改预训练路径 - 逐行精确替换
swin_path = os.path.join(CODE_DST, 'swinv2unet.py')
with open(swin_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if '.pth' in line and 'swinv2_' in line:
        # 保留原始缩进
        indent = len(line) - len(line.lstrip())
        spaces = line[:indent]
        
        if "'tiny'" in line and ':' in line:
            line = f"{spaces}'tiny':  '{PRETRAIN}/swinv2_tiny_patch4_window16_256.pth',\n"
        elif "'small'" in line and ':' in line:
            line = f"{spaces}'small': '{PRETRAIN}/swinv2_small_patch4_window16_256.pth',\n"
        elif "'base'" in line and ':' in line and 'torch.load' not in line:
            line = f"{spaces}'base':  '{PRETRAIN}/swinv2_base_patch4_window12to16_192to256_22kto1k_ft.pth',\n"
        elif 'torch.load' in line or 'checkpoint' in line:
            # 保留原始缩进和注释
            line = f"{spaces}checkpoint=torch.load('{PRETRAIN}/swinv2_base_patch4_window12to16_192to256_22kto1k_ft.pth')[\"model\"]        # 加载预训练的权重\n"
    new_lines.append(line)

with open(swin_path, 'w', encoding='utf-8') as f:
    f.writelines(new_lines)

# 验证
with open(swin_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        if 'swinv2_' in line and '.pth' in line:
            print(f'L{i}: {repr(line[:80])}'  )

from swinv2unet import Swin_LCSRB_DeformablePSP_FPNPAN
from MyDataset import MyDataset
import metrics
print('Import OK')

In [ ]:
# Cell 5: 加载模型 + 权重
NUM_CLASSES = 5
IN_CHANNELS = 5
CLASS_NAMES_CN = ['背景', '皱脊', '月溪', '断层', '地堑']
CLASS_NAMES = ['Background', 'Wrinkle Ridge', 'Rille', 'Fault', 'Graben']

CLASS_COLORS = np.array([
    [0, 0, 0],
    [255, 0, 0],
    [0, 100, 255],
    [0, 200, 0],
    [255, 255, 0],
], dtype=np.uint8)

model = Swin_LCSRB_DeformablePSP_FPNPAN(
    size=MODEL_SIZE, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS, pretrained=False
).to(device)

state = torch.load(WEIGHT_PATH, map_location=device)
model.load_state_dict(state)
model.eval()
print(f'Model ({MODEL_SIZE}) loaded from {WEIGHT_PATH}')

In [ ]:
# Cell 6: 在测试集上计算完整指标
test_data = MyDataset(images_dir=TEST_IMG, masks_dir=TEST_MASK)
test_iter = DataLoader(test_data, batch_size=1, shuffle=False, num_workers=2)

hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.float64)
all_preds = []
all_labels = []
all_names = []

with torch.no_grad(), torch.amp.autocast('cuda'):
    for img, label, name in test_iter:
        img = img.to(device)
        logits = model(img)
        pred = logits.argmax(dim=1).cpu()
        hist += metrics.multiclass_confusion(pred, label, NUM_CLASSES).double()
        all_preds.append(pred.squeeze(0).numpy())
        all_labels.append(label.squeeze(0).numpy())
        all_names.append(name[0])

m = metrics.metrics_from_hist(hist)
print('=' * 60)
print(f"Overall Accuracy: {m['accuracy']:.4f}")
print(f"Mean IoU:         {m['miou']:.4f}")
print('=' * 60)
print(f"{'Class':<16} {'IoU':>8} {'Prec':>8} {'Recall':>8} {'F1':>8}")
print('-' * 50)
for i in range(NUM_CLASSES):
    # 同时打印中英文
    label_str = f'{CLASS_NAMES_CN[i]}({CLASS_NAMES[i]})'
    print(f"{label_str:<16} {m['iou_per_class'][i]:>8.4f} {m['precision_per_class'][i]:>8.4f} {m['recall_per_class'][i]:>8.4f} {m['f1_per_class'][i]:>8.4f}")
print('-' * 50)
print(f"{'Mean':<16} {m['miou']:>8.4f} {np.mean(m['precision_per_class']):>8.4f} {np.mean(m['recall_per_class']):>8.4f} {np.mean(m['f1_per_class']):>8.4f}")

In [ ]:
# Cell 7: 训练 Loss / mIoU 曲线 (从 history.json 读取)
import json

if HISTORY_PATH and os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, 'r', encoding='utf-8') as f:
        history = json.load(f)

    ep = history['epoch']
    CLASS_COLORS_PLT = ['black', 'red', 'dodgerblue', 'green', 'orange']

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # (1) Loss
    axes[0, 0].plot(ep, history['train_loss'], 'o-', ms=3, label='Train')
    axes[0, 0].plot(ep, history['test_loss'],  's-', ms=3, label='Test')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curve'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    # (2) mIoU
    axes[0, 1].plot(ep, history['train_miou'], 'o-', ms=3, label='Train')
    axes[0, 1].plot(ep, history['test_miou'],  's-', ms=3, label='Test')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('mIoU')
    axes[0, 1].set_title('mIoU Curve'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # (3) Train per-class IoU
    if 'train_iou_per_class' in history:
        arr = np.array(history['train_iou_per_class'])
        for c in range(min(NUM_CLASSES, arr.shape[1])):
            axes[1, 0].plot(ep, arr[:, c], 'o-', ms=2, lw=1.5,
                            color=CLASS_COLORS_PLT[c], label=CLASS_NAMES[c])
        axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('IoU')
        axes[1, 0].set_title('Train Per-Class IoU'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    # (4) Test per-class IoU
    if 'test_iou_per_class' in history:
        arr = np.array(history['test_iou_per_class'])
        for c in range(min(NUM_CLASSES, arr.shape[1])):
            axes[1, 1].plot(ep, arr[:, c], 's-', ms=2, lw=1.5,
                            color=CLASS_COLORS_PLT[c], label=CLASS_NAMES[c])
        axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('IoU')
        axes[1, 1].set_title('Test Per-Class IoU'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
    plt.show()

    best_idx = int(np.argmax(history['test_miou']))
    print(f'Best test mIoU: {history["test_miou"][best_idx]:.4f} @ epoch {history["epoch"][best_idx]}')
    print(f'Final train loss: {history["train_loss"][-1]:.4f}, test loss: {history["test_loss"][-1]:.4f}')
else:
    print('未找到 history.json, 跳过 loss 曲线绘制')

In [ ]:
# Cell 7: 混淆矩阵热力图
hist_np = hist.numpy()
# 归一化为百分比 (每行除以行和)
row_sums = hist_np.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
hist_norm = hist_np / row_sums * 100

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(hist_norm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('Ground Truth')
ax.set_title('Confusion Matrix (%)')

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = hist_norm[i, j]
        color = 'white' if val > 50 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# Cell 8: 各类别 IoU 柱状图
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#333333', '#FF0000', '#0064FF', '#00C800', '#FFFF00']
bars = ax.bar(CLASS_NAMES, m['iou_per_class'], color=colors, edgecolor='black')
ax.set_ylabel('IoU')
ax.set_title(f'Per-Class IoU (mIoU={m["miou"]:.4f})')
ax.set_ylim(0, 1.0)
ax.axhline(y=m['miou'], color='gray', linestyle='--', label=f'mIoU={m["miou"]:.4f}')
for bar, val in zip(bars, m['iou_per_class']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'iou_per_class.png'), dpi=150)
plt.show()

In [ ]:
# Cell 9: 辅助函数 - 将 mask 转为彩色图
def mask_to_color(mask, class_colors=CLASS_COLORS):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(len(class_colors)):
        rgb[mask == c] = class_colors[c]
    return rgb

def get_legend_patches():
    patches = []
    for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        patches.append(mpatches.Patch(color=color/255.0, label=f'{name} (IoU={m["iou_per_class"][i]:.3f})'))
    return patches

In [ ]:
# Cell 10: 可视化 - 找有最多前景像素的样本
# 选出包含各类别最多的样本, 方便观察
fg_counts = []
for i, lbl in enumerate(all_labels):
    fg = np.sum(lbl > 0)
    fg_counts.append(fg)

# 选前景最多的 12 张
top_indices = np.argsort(fg_counts)[::-1][:12]

fig, axes = plt.subplots(4, 6, figsize=(24, 16))
for row in range(4):
    idx = top_indices[row * 1 + row]  # 每隔几张取一张
    if row < len(top_indices):
        idx = top_indices[row]
    
    pred_rgb = mask_to_color(all_preds[idx])
    label_rgb = mask_to_color(all_labels[idx])
    
    # 错误图: 正确=绿, 错误=红
    error_map = np.zeros((*all_labels[idx].shape, 3), dtype=np.uint8)
    correct = all_preds[idx] == all_labels[idx]
    fg_mask = all_labels[idx] > 0
    error_map[fg_mask & correct] = [0, 200, 0]   # 前景正确 - 绿
    error_map[fg_mask & ~correct] = [255, 0, 0]   # 前景错误 - 红
    error_map[~fg_mask & ~correct] = [255, 165, 0] # 背景误检 - 橙
    
    # WAC 原图 (第0通道)
    test_img_data = test_data[idx][0].numpy()
    wac = test_img_data[0]  # 第一个通道 WAC
    wac = (wac - wac.min()) / (wac.max() - wac.min() + 1e-8)
    
    axes[row, 0].imshow(wac, cmap='gray')
    axes[row, 0].set_title(f'WAC - {all_names[idx]}')
    axes[row, 1].imshow(label_rgb)
    axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred_rgb)
    axes[row, 2].set_title('Prediction')
    axes[row, 3].imshow(error_map)
    axes[row, 3].set_title('Error (R=miss, O=false alarm)')
    
    # DEM (第1通道)
    dem = test_img_data[1]
    dem = (dem - dem.min()) / (dem.max() - dem.min() + 1e-8)
    axes[row, 4].imshow(dem, cmap='terrain')
    axes[row, 4].set_title('DEM')
    
    # Slope (第2通道)
    slope = test_img_data[2]
    slope = (slope - slope.min()) / (slope.max() - slope.min() + 1e-8)
    axes[row, 5].imshow(slope, cmap='hot')
    axes[row, 5].set_title('Slope')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Test Predictions (top foreground samples)', fontsize=16)
fig.legend(handles=get_legend_patches(), loc='lower center', ncol=5, fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig(os.path.join(SAVE_DIR, 'predictions_top_fg.png'), dpi=150)
plt.show()

In [ ]:
# Cell 11: 每个类别单独分析 - 找该类别最多的样本
for cls_id in range(1, NUM_CLASSES):  # 跳过背景
    cls_counts = [np.sum(lbl == cls_id) for lbl in all_labels]
    top3 = np.argsort(cls_counts)[::-1][:3]
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle(f'{CLASS_NAMES[cls_id]} (class {cls_id}) - Top 3 samples', fontsize=14)
    
    for row, idx in enumerate(top3):
        gt = all_labels[idx]
        pr = all_preds[idx]
        
        # 该类别的 GT mask
        gt_cls = (gt == cls_id).astype(np.uint8)
        pr_cls = (pr == cls_id).astype(np.uint8)
        
        # TP/FP/FN 图
        tp_fp_fn = np.zeros((*gt.shape, 3), dtype=np.uint8)
        tp = (gt_cls == 1) & (pr_cls == 1)
        fp = (gt_cls == 0) & (pr_cls == 1)
        fn = (gt_cls == 1) & (pr_cls == 0)
        tp_fp_fn[tp] = [0, 200, 0]     # TP - 绿
        tp_fp_fn[fp] = [255, 165, 0]   # FP - 橙
        tp_fp_fn[fn] = [255, 0, 0]     # FN - 红
        
        wac = test_data[idx][0].numpy()[0]
        wac = (wac - wac.min()) / (wac.max() - wac.min() + 1e-8)
        
        axes[row, 0].imshow(wac, cmap='gray')
        axes[row, 0].set_title(f'WAC - {all_names[idx]}')
        axes[row, 1].imshow(gt_cls, cmap='gray')
        axes[row, 1].set_title(f'GT ({np.sum(gt_cls)} px)')
        axes[row, 2].imshow(pr_cls, cmap='gray')
        axes[row, 2].set_title(f'Pred ({np.sum(pr_cls)} px)')
        axes[row, 3].imshow(tp_fp_fn)
        axes[row, 3].set_title('G=TP, O=FP, R=FN')
    
    for ax in axes.flat:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'class_{cls_id}_{CLASS_NAMES[cls_id]}.png'), dpi=150)
    plt.show()

In [ ]:
# Cell 12: 统计每张图的 IoU, 找出最差的样本
per_image_miou = []
for i in range(len(all_preds)):
    h = metrics.multiclass_confusion(
        torch.tensor(all_preds[i]).unsqueeze(0),
        torch.tensor(all_labels[i]).unsqueeze(0),
        NUM_CLASSES
    ).double()
    im = metrics.metrics_from_hist(h)
    per_image_miou.append(im['miou'])

per_image_miou = np.array(per_image_miou)

# mIoU 分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(per_image_miou, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(per_image_miou), color='red', linestyle='--', label=f'mean={np.mean(per_image_miou):.4f}')
axes[0].set_xlabel('mIoU')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-image mIoU Distribution')
axes[0].legend()

# 最差的 10 张
worst_10 = np.argsort(per_image_miou)[:10]
axes[1].barh(range(10), per_image_miou[worst_10], color='salmon', edgecolor='black')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([all_names[i] for i in worst_10], fontsize=8)
axes[1].set_xlabel('mIoU')
axes[1].set_title('Worst 10 images')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'miou_distribution.png'), dpi=150)
plt.show()

print('\nWorst 5 images:')
for i in worst_10[:5]:
    print(f'  {all_names[i]}: mIoU={per_image_miou[i]:.4f}')

In [ ]:
# Cell 13: 可视化最差的 4 张
worst4 = np.argsort(per_image_miou)[:4]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for row, idx in enumerate(worst4):
    wac = test_data[idx][0].numpy()[0]
    wac = (wac - wac.min()) / (wac.max() - wac.min() + 1e-8)
    
    pred_rgb = mask_to_color(all_preds[idx])
    label_rgb = mask_to_color(all_labels[idx])
    
    error_map = np.zeros((*all_labels[idx].shape, 3), dtype=np.uint8)
    correct = all_preds[idx] == all_labels[idx]
    fg_mask = all_labels[idx] > 0
    error_map[fg_mask & correct] = [0, 200, 0]
    error_map[fg_mask & ~correct] = [255, 0, 0]
    error_map[~fg_mask & ~correct] = [255, 165, 0]
    
    axes[row, 0].imshow(wac, cmap='gray')
    axes[row, 0].set_title(f'WAC - {all_names[idx]}\nmIoU={per_image_miou[idx]:.4f}')
    axes[row, 1].imshow(label_rgb)
    axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred_rgb)
    axes[row, 2].set_title('Prediction')
    axes[row, 3].imshow(error_map)
    axes[row, 3].set_title('Error')

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Worst 4 Predictions', fontsize=16)
fig.legend(handles=get_legend_patches(), loc='lower center', ncol=5, fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig(os.path.join(SAVE_DIR, 'worst_predictions.png'), dpi=150)
plt.show()

In [ ]:
# Cell 15: 导出全部 304 张测试图预测 (pred_mask + pred_vis)
from PIL import Image
from MyDataset import CHANNEL_MEAN, CHANNEL_STD

mask_dir = os.path.join(PRED_DIR, 'pred_mask')
vis_dir  = os.path.join(PRED_DIR, 'pred_vis')
os.makedirs(mask_dir, exist_ok=True)
os.makedirs(vis_dir, exist_ok=True)

def error_map_fn(gt, pred):
    out = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)
    gt_fg, pr_fg = gt > 0, pred > 0
    out[gt_fg & pr_fg]    = [0, 200, 0]    # TP
    out[gt_fg & (~pr_fg)] = [255, 0, 0]    # FN
    out[(~gt_fg) & pr_fg] = [255, 165, 0]  # FP
    return out

print(f'导出 {len(all_preds)} 张测试图...')
for i in range(len(all_preds)):
    stem = all_names[i]
    pred_np = all_preds[i].astype(np.uint8)
    gt_np   = all_labels[i].astype(np.uint8)

    # 灰度 mask
    Image.fromarray(pred_np, mode='L').save(os.path.join(mask_dir, f'{stem}.png'))

    # 四合一可视化: WAC / GT / Pred / Error
    x = test_data[i][0].numpy()
    wac = x[0] * CHANNEL_STD[0] + CHANNEL_MEAN[0]
    wac = np.clip(wac, 0, 1)

    gt_rgb   = mask_to_color(gt_np)
    pred_rgb = mask_to_color(pred_np)
    err_rgb  = error_map_fn(gt_np, pred_np)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(wac, cmap='gray');       axes[0].set_title(f'WAC - {stem}', fontsize=7)
    axes[1].imshow(gt_rgb);                 axes[1].set_title('GT')
    axes[2].imshow(pred_rgb);               axes[2].set_title('Pred')
    axes[3].imshow(err_rgb);                axes[3].set_title('Error(G=TP,R=FN,O=FP)')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f'{stem}.png'), dpi=100)
    plt.close(fig)

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(all_preds)} done')

print(f'\n导出完成!')
print(f'  pred_mask: {len(os.listdir(mask_dir))} 张 → {mask_dir}')
print(f'  pred_vis:  {len(os.listdir(vis_dir))} 张 → {vis_dir}')

In [ ]:
# Cell 16: 打包所有结果 + 总结
import zipfile

# 打包 visualization + all_predictions
zip_path = '/kaggle/working/eval_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [SAVE_DIR, PRED_DIR]:
        for root, dirs, files in os.walk(folder):
            for f in files:
                fpath = os.path.join(root, f)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
print(f'打包完成: {zip_path}')
!ls -lh /kaggle/working/eval_results.zip

print('\n' + '=' * 60)
print('STDL-Net Small - Test Results Summary')
print('=' * 60)
print(f'Overall Accuracy: {m["accuracy"]:.4f}')
print(f'Mean IoU:         {m["miou"]:.4f}')
print()
for i in range(NUM_CLASSES):
    print(f'{CLASS_NAMES[i]}: IoU={m["iou_per_class"][i]:.4f}  F1={m["f1_per_class"][i]:.4f}  Prec={m["precision_per_class"][i]:.4f}  Recall={m["recall_per_class"][i]:.4f}')
print()
print(f'Visualizations: {SAVE_DIR} ({len(os.listdir(SAVE_DIR))} files)')
print(f'All predictions: {PRED_DIR}')
print(f'  pred_mask: {len(os.listdir(os.path.join(PRED_DIR, "pred_mask")))} images')
print(f'  pred_vis:  {len(os.listdir(os.path.join(PRED_DIR, "pred_vis")))} images')
print(f'\n下载 eval_results.zip 即可获取全部结果')